# Mini Project 1 — Metadata Completeness in The Met Collection

**Your name:** *(fill in)*  
**Dataset:** The Metropolitan Museum of Art Collection API  
**Date:** May 2026  

This notebook follows the Week 6 MP1 structure. Run cells from top to bottom. The setup cell installs **pandas**, **plotly**, **kaleido**, **requests**, and **nbformat** if needed.

**Note:** Fetching the sample from the API takes several minutes (500 requests with delays). After `met_metadata_random_sample.csv` exists in the **MP1** folder, re-runs skip downloading unless you set `FORCE_REFRESH = True`.


In [ ]:
# Setup — run this cell first
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat", "requests"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import random
import time
from pathlib import Path

import pandas as pd
import plotly.express as px
import requests

# Run this notebook from the MP1 folder so paths resolve here
DATA_DIR = Path.cwd()
CSV_PATH = DATA_DIR / "met_metadata_random_sample.csv"
PNG1 = DATA_DIR / "missing_metadata_by_field.png"
PNG2 = DATA_DIR / "completeness_by_department.png"
PNG3 = DATA_DIR / "completeness_score_distribution.png"

print("Setup complete. Working directory:", DATA_DIR.resolve())


---

## Section 1 — Overview

**Dataset:** The Metropolitan Museum of Art publishes its collection through the **Collection API** ([Met Museum Open Access API](https://metmuseum.github.io/)). Each object has a stable `objectID` and a JSON record with catalog fields such as title, department, culture, medium, dates, and attribution.

**Why this dataset:** Museum metadata supports search, interpretation, and digital access. Understanding **which fields are often missing** helps prioritize cataloging effort and sets expectations for interfaces or research built on the API.

**Three analytical questions:**

1. **Q1:** Which metadata fields are most often missing in the random sample?
2. **Q2:** Which departments have higher or lower average metadata completeness?
3. **Q3:** How complete are individual object records overall (distribution of completeness scores)?

**What a practitioner would do with these findings:** Collection managers and UX designers can target enrichment on the weakest fields and departments, and communicate limits when designing search filters or studies that rely on specific facets.


---

## Section 2 — Data Profile

We use a **reproducible random sample of 500 object IDs** (`seed=42`) from the full ID list, then **detail records only for those IDs** (not the full collection). Columns: `objectID`, `title`, `department`, `objectDate`, `culture`, `medium`, `classification`, `artistDisplayName`, `country`, `period`, `objectBeginDate`, `objectEndDate`.

**Missing values:** Absent JSON keys, `None`, and empty or whitespace-only strings are treated as missing. Parsing uses `.get()` for safe access.

Reflect after the code cells:

- How many rows and columns? What does each column represent?
- Any data quality issues (sparse fields, uneven departments)?
- Which fields drive the completeness metrics?


In [ ]:
# --- Build sample from API (slow first time) ---
FORCE_REFRESH = False

API_BASE = "https://collectionapi.metmuseum.org/public/collection/v1"
FIELD_KEYS = [
    "objectID",
    "title",
    "department",
    "objectDate",
    "culture",
    "medium",
    "classification",
    "artistDisplayName",
    "country",
    "period",
    "objectBeginDate",
    "objectEndDate",
]
DETAIL_KEYS = [k for k in FIELD_KEYS if k != "objectID"]


def fetch_json(url, max_retries=6, sleep_base=0.8):
    """GET with retries for rate limits (403/429) and server errors."""
    for attempt in range(max_retries):
        try:
            r = requests.get(
                url,
                timeout=45,
                headers={"User-Agent": "HCDE530-MP1-metadata-study/1.0 (educational)"},
            )
            if r.status_code == 404:
                return None
            if r.status_code in (403, 429, 500, 502, 503, 504):
                if attempt == max_retries - 1:
                    return None
                wait = sleep_base * (2**attempt) + random.uniform(0, 0.5)
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if attempt == max_retries - 1:
                return None
            time.sleep(sleep_base * (2**attempt))
    return None


def row_from_detail(j, oid):
    """Safe .get(); empty strings treated as missing (None)."""
    if not j or not isinstance(j, dict):
        return {"objectID": oid, **{k: None for k in DETAIL_KEYS}}
    out = {"objectID": j.get("objectID", oid)}
    for k in DETAIL_KEYS:
        v = j.get(k)
        if v is None:
            out[k] = None
        elif isinstance(v, str) and v.strip() == "":
            out[k] = None
        else:
            out[k] = v
    return out


def build_sample_csv():
    print("Fetching full object ID list...")
    listing = fetch_json(f"{API_BASE}/objects")
    all_ids = listing["objectIDs"]
    random.seed(42)
    sample_ids = random.sample(all_ids, 500)
    print(f"Sampled {len(sample_ids)} IDs (seed=42). Fetching details...")
    rows = []
    for i, oid in enumerate(sample_ids, 1):
        j = fetch_json(f"{API_BASE}/objects/{oid}")
        rows.append(row_from_detail(j, oid))
        if i % 50 == 0:
            print(f"  ... {i}/500")
        time.sleep(0.25 + random.uniform(0, 0.15))
    df_new = pd.DataFrame(rows, columns=FIELD_KEYS)
    df_new.to_csv(CSV_PATH, index=False)
    print("Saved:", CSV_PATH.resolve(), df_new.shape)


if FORCE_REFRESH or not CSV_PATH.exists():
    build_sample_csv()
else:
    print("Using existing CSV:", CSV_PATH.resolve(), "(set FORCE_REFRESH = True to rebuild)")


In [ ]:
def _normalize_missing(val):
    if pd.isna(val):
        return pd.NA
    if isinstance(val, str) and val.strip() == "":
        return pd.NA
    return val


df = pd.read_csv(CSV_PATH)
for col in df.columns:
    if col != "objectID":
        df[col] = df[col].apply(_normalize_missing)

print(df.shape)
df.head()


In [ ]:
df.info()


**Data profile notes**

- **500 rows × 12 columns:** one random object per row; `objectID` is the key; the other eleven columns are catalog metadata from the API.
- **Quality:** Missingness is expected for optional facets (e.g. country, period). `objectID`, `title`, and `department` are almost always present in practice.
- **Analysis focus:** Eleven content fields (all except `objectID`) define a **completeness score** = share of those fields that are non-missing.


---

## Section 3 — Analysis


**Question 1:** Which metadata fields are most often missing in the random sample?


In [ ]:
META_FIELDS = [
    "title",
    "department",
    "objectDate",
    "culture",
    "medium",
    "classification",
    "artistDisplayName",
    "country",
    "period",
    "objectBeginDate",
    "objectEndDate",
]

missing_rows = []
for field in META_FIELDS:
    pct = 100.0 * df[field].isna().sum() / len(df)
    missing_rows.append({"field": field, "pct_missing": pct})

missing_by_field = pd.DataFrame(missing_rows).sort_values("pct_missing", ascending=False)
missing_by_field


**Interpretation (Q1):** The ranked table shows which facets are sparse in this draw. Fields with high `%` missing are risky to use as required filters without imputation or alternate labels.


**Question 2:** Which departments have higher or lower average metadata completeness?


In [ ]:
present_mask = df[META_FIELDS].notna()
df["completeness_score"] = present_mask.sum(axis=1) / len(META_FIELDS) * 100.0

by_dept = (
    df.groupby("department", dropna=False)["completeness_score"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "mean_completeness_pct", "count": "n_objects"})
    .sort_values("mean_completeness_pct", ascending=False)
)
by_dept


**Interpretation (Q2):** Higher mean completeness suggests more facets are routinely filled for objects in that department in this sample; lower means may reflect object type (anonymous works) or historical cataloging—not necessarily “worse” curating.


**Question 3:** How complete are individual object records overall?


In [ ]:
df["completeness_score"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])


**Interpretation (Q3):** Scores run from 0–100%. The spread shows whether most records are “almost full” or whether many objects have several empty facets—important for designing multi-field search.


---

## Section 4 — Visualization

Plotly figures; each chart is saved as PNG in this folder.


In [ ]:
m1 = missing_by_field.sort_values("pct_missing", ascending=True)
fig1 = px.bar(
    m1,
    x="pct_missing",
    y="field",
    orientation="h",
    title="Missing Metadata Percentage by Field",
    labels={"pct_missing": "Percent of records missing", "field": "Metadata field"},
)
fig1.update_layout(yaxis={"categoryorder": "total ascending"}, height=480, margin=dict(l=130))
fig1.write_image(str(PNG1), width=900, height=520, scale=2)
fig1.show()

dept_plot = (
    df.groupby("department", dropna=False)["completeness_score"]
    .mean()
    .reset_index()
    .sort_values("completeness_score", ascending=True)
)
h2 = max(520, 22 * len(dept_plot) + 80)
fig2 = px.bar(
    dept_plot,
    x="completeness_score",
    y="department",
    orientation="h",
    title="Average Metadata Completeness by Department",
    labels={"completeness_score": "Mean completeness (%)", "department": "Department"},
)
fig2.update_layout(height=h2, margin=dict(l=200, r=40))
fig2.write_image(str(PNG2), width=950, height=h2, scale=2)
fig2.show()

fig3 = px.histogram(
    df,
    x="completeness_score",
    nbins=25,
    title="Distribution of Metadata Completeness Scores Across Sampled Objects",
    labels={
        "completeness_score": "Completeness score (% of fields present)",
        "count": "Number of objects",
    },
)
fig3.update_layout(bargap=0.05)
fig3.write_image(str(PNG3), width=900, height=500, scale=2)
fig3.show()

print("Saved:", PNG1.name, PNG2.name, PNG3.name)


**Chart rationale**

- **missing_metadata_by_field.png:** Horizontal bar chart—each bar is a metadata field; length is percent missing. Good for many category labels and answers Q1 directly.
- **completeness_by_department.png:** Horizontal bar of mean completeness by department—compares groups (Q2). Sorted low-to-high to spotlight departments with lower average fill rates.
- **completeness_score_distribution.png:** Histogram of per-object completeness (Q3). Shows central tendency and tails for how “full” records feel in the sample.


---

## Section 5 — Conclusions


**Summary of findings**

This notebook analyzed **metadata completeness** for a **reproducible random sample of 500** The Met object IDs (`seed=42`), fetching **details only for those IDs**. Missingness is **uneven across fields**: a few facets are often blank while core fields like title and department are usually filled. **Average completeness by department** differs, which may reflect object type and catalog history rather than a single “quality score.” At the **object level**, completeness scores show how often records have most facets populated—useful for scoping search UX. **Limitations:** one sample in time; API throttling can occasionally fail a row (retries mitigate this); completeness does not measure factual accuracy of filled values.

---

**Competency claims:** See **`week6.md`** in this folder for C3–C7 statements.
